<a href="https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/problem_notebooks/Problem03_Student_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>&nbsp;&nbsp;<a href="https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/kadimetla/AIML_Learning_Gang/main/statistics/problem_notebooks/Problem03_Student_Performance.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

**Compatible with Google Colab, Kaggle, JupyterLab, VS Code, and Binder.**

| Platform | How to open |
|----------|-------------|
| **Google Colab** | Click the Colab badge above |
| **Kaggle** | Click the Kaggle badge above, or *New Notebook → File → Import Notebook* |
| **JupyterLab** | `pip install jupyterlab && jupyter lab` in your terminal |
| **VS Code** | Open with the Jupyter extension (`.ipynb` support built-in) |
| **Binder** | Visit mybinder.org and point it to this repo |

> **Requirements:** `numpy` `pandas` `matplotlib` `seaborn` `scikit-learn` `scipy`  
> Pre-installed on Colab and Kaggle. The next cell installs them if missing.


In [ ]:
# Install required packages — safe on Colab, Kaggle, or local environments
%pip install numpy pandas matplotlib seaborn scikit-learn scipy --quiet


# 🎓 Problem 03: Predicting Student Performance

**Story:** A school wants to predict which students might struggle before the final exam, so teachers can offer extra help early. What data can tell us?

This notebook teaches statistics and machine learning concepts through a realistic, relatable problem — student performance prediction.

**Concepts covered:** Descriptive statistics, distributions, correlation, feature engineering, regression, classification, model evaluation, and the full data science workflow.

## 📦 Section 0: Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Color scheme
PASS_COLOR = '#2ecc71'   # green — passing (score >= 60)
FAIL_COLOR = '#e74c3c'   # red   — failing  (score < 60)
ACCENT     = '#3498db'   # blue  — general accent

print('All imports successful.')

## 🏫 Section 1: Meet the Problem

### The Scenario

A school counselor wants to **predict which students are at risk of failing the final exam** — before it happens. If we can flag struggling students early, teachers can intervene: extra tutoring, check-ins, or adjusted workloads.

### Our Goal
1. **Regression:** Predict a student's actual `final_score` (0–100)
2. **Classification:** Predict whether a student will **pass** (score ≥ 60) or **fail** (score < 60)

### Why This Matters
Early identification changes outcomes. A student flagged at week 4 can get help at week 5 — not after they've already failed.

### What Features Do We Have?
We collected data on **500 students** covering study habits, attendance, sleep, family background, and more. Let's explore.

In [ ]:
# ── Generate synthetic but realistic student data ──────────────────────────
np.random.seed(42)
n = 500

# Study hours per week (right-skewed, 0-40 hours)
study_hours = np.random.gamma(2, 5, n).clip(0, 40)

# Attendance rate (left-skewed, most students attend most classes)
attendance = np.random.beta(8, 2, n) * 100  # 0-100%

# Sleep hours per night (approximately normal, 4-10 hours)
sleep_hours = np.random.normal(7, 1.2, n).clip(4, 10)

# Previous GPA (0-4.0 scale)
prev_gpa = np.random.beta(5, 2, n) * 4

# Parental education level (0=No HS, 1=HS, 2=Some College, 3=Bachelor, 4=Graduate)
parental_edu = np.random.choice([0,1,2,3,4], n, p=[0.05,0.20,0.25,0.30,0.20])

# Part-time job hours per week (many students have 0)
part_time_hours = np.where(
    np.random.random(n) < 0.4,
    0,
    np.random.gamma(2, 6, n).clip(0, 30)
)

# Extracurricular activities (0-5 activities)
extracurricular = np.random.poisson(1.5, n).clip(0, 5)

# Distance from school (km)
distance_km = np.random.exponential(8, n).clip(0.5, 50)

# Internet quality at home (1-5 scale)
internet_quality = np.random.choice([1,2,3,4,5], n, p=[0.05,0.10,0.25,0.35,0.25])

# Final exam score (target) — influenced by above factors
noise = np.random.normal(0, 8, n)
score = (
    15 +
    study_hours * 1.2 +
    attendance * 0.3 +
    sleep_hours * 2.5 +
    prev_gpa * 12 +
    parental_edu * 2.5 +
    internet_quality * 1.5 -
    part_time_hours * 0.4 +
    extracurricular * 1.0 -
    distance_km * 0.1 +
    noise
).clip(0, 100)

df = pd.DataFrame({
    'study_hours':     study_hours,
    'attendance_pct':  attendance,
    'sleep_hours':     sleep_hours,
    'prev_gpa':        prev_gpa,
    'parental_edu':    parental_edu,
    'part_time_hours': part_time_hours,
    'extracurricular': extracurricular,
    'distance_km':     distance_km,
    'internet_quality':internet_quality,
    'final_score':     score
})

# Add realistic missing values
np.random.seed(42)
df.loc[np.random.choice(n, 30, replace=False), 'sleep_hours']     = np.nan
df.loc[np.random.choice(n, 15, replace=False), 'distance_km']     = np.nan
df.loc[np.random.choice(n, 20, replace=False), 'part_time_hours'] = np.nan

print(f'Dataset shape: {df.shape[0]} students, {df.shape[1]} features')
df.head(10)

### Data Dictionary

| Feature | Meaning | Range | Type |
|---|---|---|---|
| `study_hours` | Hours spent studying per week | 0 – 40 | Continuous |
| `attendance_pct` | Percentage of classes attended | 0 – 100% | Continuous |
| `sleep_hours` | Average sleep hours per night | 4 – 10 | Continuous |
| `prev_gpa` | GPA from previous semester | 0.0 – 4.0 | Continuous |
| `parental_edu` | Highest parental education level (0=No HS, 1=HS, 2=Some College, 3=Bachelor, 4=Graduate) | 0 – 4 | Ordinal |
| `part_time_hours` | Hours working a part-time job per week | 0 – 30 | Continuous |
| `extracurricular` | Number of extracurricular activities | 0 – 5 | Count |
| `distance_km` | Distance from home to school in km | 0.5 – 50 | Continuous |
| `internet_quality` | Home internet quality (1=Poor, 5=Excellent) | 1 – 5 | Ordinal |
| `final_score` | **Target:** Final exam score | 0 – 100 | Continuous |

## 🩺 Section 2: Data Health Check

**"Is our data trustworthy before we analyze it?"**

Before any analysis, we need to understand: How complete is the data? Are there missing values? What are the basic statistics?

In [ ]:
# ── Basic descriptive statistics ───────────────────────────────────────────
print('=== Descriptive Statistics ===')
print('\nFor each column, this shows:')
print('  count  — how many non-missing values')
print('  mean   — the average')
print('  std    — standard deviation (spread around the mean)')
print('  min/max — extreme values')
print('  25%/50%/75% — quartiles (25%=Q1, 50%=median, 75%=Q3)')
print()
df.describe().round(2)

In [ ]:
# ── Missing value analysis ─────────────────────────────────────────────────
missing = pd.DataFrame({
    'count_missing': df.isnull().sum(),
    'pct_missing':   (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['count_missing'] > 0]

print('=== Missing Values ===')
print(missing.to_string())
print()
print('Columns with missing data:')
for col, row in missing.iterrows():
    print(f'  {col}: {int(row["count_missing"])} missing ({row["pct_missing"]}%)')

In [ ]:
# ── Missing value heatmap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap: gray = present, red = missing
missing_mask = df.isnull()
sns.heatmap(
    missing_mask.T,
    ax=axes[0],
    cmap=['#95a5a6', '#e74c3c'],   # gray = present, red = missing
    cbar_kws={'label': 'Missing (red)'},
    yticklabels=df.columns,
    xticklabels=False
)
axes[0].set_title('Missing Value Map\n(red = missing, gray = present)', fontsize=12)
axes[0].set_xlabel('Student Index')

# Bar chart of missing counts
missing_pct = df.isnull().mean() * 100
missing_pct[missing_pct > 0].plot.barh(ax=axes[1], color=FAIL_COLOR, alpha=0.8)
axes[1].set_xlabel('% Missing')
axes[1].set_title('Percentage of Missing Values\nby Column', fontsize=12)
for i, v in enumerate(missing_pct[missing_pct > 0]):
    axes[1].text(v + 0.1, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.show()

print('\nsleep_hours and distance_km have missing data — we need to decide how to handle this.')
print('part_time_hours also has some missing values.')

### 📊 CONCEPT: Strategies for Missing Data

When data is missing, we have three main choices:

1. **Delete rows** — simplest, but risky if data is not "missing at random" (we might lose a biased subset)
2. **Fill with mean/median** — common and fast; **median is safer than mean when data is skewed** because the mean is pulled by outliers
3. **Predict missing values** — use a model to fill in missing data based on other features (most sophisticated)

For this dataset: missing rates are small (3–6%), so we'll use median imputation in Section 7.

## 📊 Section 3: Column-by-Column Analysis

**"Understanding our students — one feature at a time"**

Before we build models, we need to understand each feature individually. What does each distribution look like? Are there outliers? What's normal for a student?

In [ ]:
# ── 3.1 study_hours ───────────────────────────────────────────────────────
col = 'study_hours'
data = df[col].dropna()

mean_val   = data.mean()
median_val = data.median()
std_val    = data.std()
skew_val   = data.skew()
pct_gt20   = (data > 20).mean() * 100

print(f'=== Study Hours per Week ===')
print(f'  Mean:              {mean_val:.2f} hrs/week')
print(f'  Median:            {median_val:.2f} hrs/week')
print(f'  Std Dev:           {std_val:.2f}')
print(f'  Min / Max:         {data.min():.1f} / {data.max():.1f}')
print(f'  Skewness:          {skew_val:.3f}  (positive = right-skewed)')
print(f'  % studying >20hrs: {pct_gt20:.1f}%')
print(f'\n  The mean ({mean_val:.1f}) > median ({median_val:.1f}) confirms right skew.')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram + KDE
axes[0].hist(data, bins=30, color=ACCENT, alpha=0.7, edgecolor='white', density=True)
data.plot.kde(ax=axes[0], color='navy', linewidth=2)
axes[0].axvline(mean_val,   color='red',    linestyle='--', linewidth=1.8, label=f'Mean {mean_val:.1f}')
axes[0].axvline(median_val, color='orange', linestyle='--', linewidth=1.8, label=f'Median {median_val:.1f}')
axes[0].set_xlabel('Study Hours per Week')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Study Hours\n(Histogram + KDE)')
axes[0].legend()

# Box plot
axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor=ACCENT, alpha=0.6))
axes[1].set_xticks([1])
axes[1].set_xticklabels(['study_hours'])
axes[1].set_ylabel('Hours per Week')
axes[1].set_title('Box Plot: Study Hours\n(shows median, IQR, outliers)')

plt.suptitle('Study Hours per Week', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nInsight: Right-skewed — most students study 5-15hrs/week, very few study >30hrs.')
print('The gap between mean and median confirms this right skew.')

In [ ]:
# ── study_hours: Distribution analysis — Gamma-like, right-skewed ───────────
from scipy import stats as sp_stats

col = 'study_hours'
data = df[col].dropna()
mean_, std_ = data.mean(), data.std()

print(f'=== {col}: Distribution Analysis ===')
print(f'  Mean:     {mean_:.4f}')
print(f'  Median:   {data.median():.4f}')
print(f'  Std Dev:  {std_:.4f}')
print(f'  Skewness: {data.skew():.4f}  (positive = right tail — most study moderate, few study a lot)')
print(f'  Kurtosis: {data.kurtosis():.4f}')
w, p = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
print(f'  Shapiro-Wilk: W={w:.4f}, p={p:.2e}  → {"NOT normal" if p<0.05 else "Normal"}')
print()
print('Distribution type: RIGHT-SKEWED (Gamma-like)')
print('study_hours ≥ 0 with a right tail — most students study moderately, a few study extensively')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(max(0, data.min()-1), data.max()+2, 300)
axes[0].hist(data, bins=35, color='#42A5F5', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0], color='black', lw=2, label='KDE (actual)')
axes[0].plot(x, sp_stats.norm.pdf(x, mean_, std_), 'r--', lw=2.5, label='Normal fit')
axes[0].set_xlabel('Study Hours per Week')
axes[0].set_title(f'study_hours  (skew={data.skew():.2f})')
axes[0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: study_hours vs Normal\n(right skew = points curve up at right)')
axes[1].get_lines()[0].set(color='#42A5F5', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('study_hours: Right-Skewed (Gamma-like Distribution)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **CONCEPT: Gamma Distribution**
>
> The **Gamma distribution** models quantities that are:
> - Always non-negative (≥ 0)
> - Right-skewed (most values are moderate, a few are very large)
> - Often measuring time, effort, or resource usage
>
> `study_hours` fits this pattern: students can't study negative hours, most study
> a moderate amount (5–15 hours/week), and a few outliers study 30+ hours.
>
> **Q-Q reading:** Points curve upward at the right end — the right tail is
> heavier than a Normal would predict. This is the signature of a Gamma-like distribution.

In [ ]:
# ── 3.2 attendance_pct ────────────────────────────────────────────────────
col  = 'attendance_pct'
data = df[col].dropna()

mean_val   = data.mean()
median_val = data.median()
pct_gt90   = (data > 90).mean() * 100
pct_lt60   = (data < 60).mean() * 100

print('=== Attendance Rate ===')
print(f'  Mean:                    {mean_val:.2f}%')
print(f'  Median:                  {median_val:.2f}%')
print(f'  % students with >90%:    {pct_gt90:.1f}%')
print(f'  % students with <60%:    {pct_lt60:.1f}% (at-risk zone)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(data, bins=30, color='#9b59b6', alpha=0.7, edgecolor='white', density=True)
data.plot.kde(ax=axes[0], color='purple', linewidth=2)
axes[0].axvline(60, color=FAIL_COLOR, linestyle=':', linewidth=2, label='60% at-risk')
axes[0].axvline(75, color='orange',   linestyle=':', linewidth=2, label='75% warning')
axes[0].axvline(90, color=PASS_COLOR, linestyle=':', linewidth=2, label='90% good')
axes[0].set_xlabel('Attendance (%)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Attendance\n(Histogram + KDE)')
axes[0].legend(fontsize=9)

axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#9b59b6', alpha=0.6))
axes[1].set_xticks([1])
axes[1].set_xticklabels(['attendance_pct'])
axes[1].set_ylabel('Attendance (%)')
axes[1].set_title('Box Plot: Attendance Rate')

plt.suptitle('Attendance Rate (%)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nInsight: Left-skewed — most students attend most classes.')
print(f'Only {pct_lt60:.1f}% of students are in the at-risk zone (<60% attendance).')

In [ ]:
# ── attendance_pct: Distribution analysis — Beta-like, left-skewed ──────────
from scipy import stats as sp_stats

col = 'attendance_pct'
data = df[col].dropna()
mean_, std_ = data.mean(), data.std()

print(f'=== {col}: Distribution Analysis ===')
print(f'  Mean:     {mean_:.4f}%')
print(f'  Median:   {data.median():.4f}%')
print(f'  Std Dev:  {std_:.4f}')
print(f'  Skewness: {data.skew():.4f}  (negative = LEFT tail — most attend >80%)')
print(f'  Kurtosis: {data.kurtosis():.4f}')
print(f'  Range:    [{data.min():.1f}%, {data.max():.1f}%]  (bounded 0–100)')
w, p = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
print(f'  Shapiro-Wilk: W={w:.4f}, p={p:.2e}  → {"NOT normal" if p<0.05 else "Normal"}')
print()
print('Distribution type: LEFT-SKEWED (Beta-like — bounded percentage, pile-up near 100%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(50, 105, 300)
axes[0].hist(data, bins=35, color='#66BB6A', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0], color='black', lw=2, label='KDE (actual)')
axes[0].plot(x, sp_stats.norm.pdf(x, mean_, std_), 'r--', lw=2.5, label='Normal fit')
axes[0].set_xlabel('Attendance (%)')
axes[0].set_title(f'attendance_pct  (skew={data.skew():.2f})')
axes[0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: attendance_pct vs Normal\n(points dip below line at left = left skew)')
axes[1].get_lines()[0].set(color='#66BB6A', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('attendance_pct: Left-Skewed (Beta-like)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **CONCEPT: Beta Distribution — Modelling Proportions**
>
> A **Beta distribution** models variables that are **bounded between 0 and 1** (or 0–100%).
> It is parameterised by two shape parameters (α, β) that control whether it piles up
> at the low end, high end, or centre.
>
> `attendance_pct` is left-skewed because most students attend regularly (pile-up near 100%).
> The tail extends leftward toward students who rarely show up.
>
> **Q-Q reading:** Points dip below the line at the lower end — the left tail is
> lighter than a Normal would predict (there's a floor at 0%, cutting off the tail).
>
> **Key contrast with study_hours:** Right-skewed vs left-skewed — two opposite shapes,
> both non-normal for the same reason (bounded variable with a natural piling-up point).

In [ ]:
# ── 3.3 sleep_hours ───────────────────────────────────────────────────────
col  = 'sleep_hours'
data = df[col].dropna()   # has missing values

mean_val   = data.mean()
median_val = data.median()
std_val    = data.std()

print(f'=== Sleep Hours per Night (Note: {df[col].isnull().sum()} missing values, using dropna) ===')
print(f'  Mean:   {mean_val:.2f} hrs')
print(f'  Median: {median_val:.2f} hrs')
print(f'  Std:    {std_val:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(data, bins=25, color='#1abc9c', alpha=0.7, edgecolor='white', density=True)
data.plot.kde(ax=axes[0], color='teal', linewidth=2)
axes[0].axvline(mean_val,   color='red',    linestyle='--', linewidth=1.8, label=f'Mean {mean_val:.1f}')
axes[0].axvline(median_val, color='orange', linestyle='--', linewidth=1.8, label=f'Median {median_val:.1f}')
axes[0].set_xlabel('Sleep Hours per Night')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Sleep Hours')
axes[0].legend()

# Box plot
axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#1abc9c', alpha=0.6))
axes[1].set_xticks([1])
axes[1].set_xticklabels(['sleep_hours'])
axes[1].set_ylabel('Hours')
axes[1].set_title('Box Plot: Sleep Hours')

# Q-Q Plot
stats.probplot(data, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot: Sleep Hours\nvs Normal Distribution')
axes[2].get_lines()[0].set(color=ACCENT, alpha=0.6)
axes[2].get_lines()[1].set(color='red', linewidth=2)

plt.suptitle('Sleep Hours per Night', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 CONCEPT — Q-Q Plot:')
print('  A Q-Q plot compares our data to a perfect normal distribution.')
print('  Points on the diagonal line = perfectly normal.')
print('  Deviations at the tails = the data is heavier or lighter-tailed than normal.')
print('\nInsight: Sleep hours is approximately normal, centered around 7 hrs.')
print('Sleep is one of the few student features that follows a normal distribution. The others are more skewed.')

In [ ]:
# ── sleep_hours: Distribution analysis — Approximately Normal ───────────────
from scipy import stats as sp_stats

col = 'sleep_hours'
data = df[col].dropna()
mean_, std_ = data.mean(), data.std()

print(f'=== {col}: Distribution Analysis ===')
print(f'  Mean:     {mean_:.4f} hours')
print(f'  Median:   {data.median():.4f} hours')
print(f'  Std Dev:  {std_:.4f}')
print(f'  Skewness: {data.skew():.4f}  (close to 0 = approximately symmetric)')
print(f'  Kurtosis: {data.kurtosis():.4f}')
print(f'  Range:    [{data.min():.1f}, {data.max():.1f}] hours  (clipped at 4–10)')
w, p = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
print(f'  Shapiro-Wilk: W={w:.4f}, p={p:.2e}  → {"NOT normal" if p<0.05 else "Normal"}')
print()
print('Distribution type: APPROXIMATELY NORMAL (generated from Normal(7, 1.2), clipped)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(3, 11, 300)
axes[0].hist(data, bins=30, color='#9E9E9E', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0], color='black', lw=2, label='KDE (actual)')
axes[0].plot(x, sp_stats.norm.pdf(x, mean_, std_), 'r--', lw=2.5,
             label=f'Normal N({mean_:.1f}, {std_:.1f}²)')
axes[0].set_xlabel('Sleep Hours per Night')
axes[0].set_title(f'sleep_hours  (skew={data.skew():.2f})')
axes[0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: sleep_hours vs Normal\n(points track close to line = approximately normal)')
axes[1].get_lines()[0].set(color='#9E9E9E', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('sleep_hours: The Most Normal Feature in This Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **sleep_hours: The Most "Normal" Feature Here**
>
> Among all 9 features, `sleep_hours` comes closest to a Normal distribution.
> The Q-Q plot tracks close to the diagonal line — points don't curve away at the ends.
>
> Why is sleep approximately normal? Human sleep needs cluster around a biological
> optimum (~7–8 hours) with natural variation above and below — this is exactly the
> kind of process that produces bell curves.
>
> The slight deviation at the tails comes from the artificial clip at 4 and 10 hours.
> Compare this Q-Q to `study_hours` (curves up at right = right skew) and
> `attendance_pct` (dips at left = left skew) — the contrast makes clear how
> different distribution shapes look on a Q-Q plot.

In [ ]:
# ── 3.4 prev_gpa ──────────────────────────────────────────────────────────
col  = 'prev_gpa'
data = df[col].dropna()

print('=== Previous GPA ===')
print(f'  Mean:   {data.mean():.3f}')
print(f'  Median: {data.median():.3f}')
print(f'  Std:    {data.std():.3f}')
print(f'  Skew:   {data.skew():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(data, bins=30, color='#e67e22', alpha=0.7, edgecolor='white', density=True)
data.plot.kde(ax=axes[0], color='darkorange', linewidth=2)
axes[0].axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.8, label=f'Mean {data.mean():.2f}')
axes[0].axvline(data.median(), color='orange', linestyle='--', linewidth=1.8, label=f'Median {data.median():.2f}')
axes[0].set_xlabel('Previous GPA (0.0 – 4.0)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Previous GPA')
axes[0].legend()

axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#e67e22', alpha=0.6))
axes[1].set_xticks([1])
axes[1].set_xticklabels(['prev_gpa'])
axes[1].set_ylabel('GPA')
axes[1].set_title('Box Plot: Previous GPA')

plt.suptitle('Previous GPA', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nInsight: Slight left-skew — higher GPAs are more common, but the distribution')
print('         is broad. Most students had 2.5–4.0 GPA previously.')

In [ ]:
# ── prev_gpa: Distribution analysis ──────────────────────────────────────────
from scipy import stats as sp_stats

col = 'prev_gpa'
data = df[col].dropna()
mean_, std_ = data.mean(), data.std()

print(f'=== {col}: Distribution Analysis ===')
print(f'  Mean:     {mean_:.4f}')
print(f'  Median:   {data.median():.4f}')
print(f'  Std Dev:  {std_:.4f}')
print(f'  Skewness: {data.skew():.4f}')
print(f'  Kurtosis: {data.kurtosis():.4f}')
print(f'  Range:    [{data.min():.2f}, {data.max():.2f}]')
w, p = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
print(f'  Shapiro-Wilk: W={w:.4f}, p={p:.2e}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(data.min()-0.2, data.max()+0.2, 300)
axes[0].hist(data, bins=30, color='#AB47BC', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0], color='black', lw=2, label='KDE')
axes[0].plot(x, sp_stats.norm.pdf(x, mean_, std_), 'r--', lw=2.5, label='Normal fit')
axes[0].set_xlabel('Previous GPA')
axes[0].set_title(f'prev_gpa  (skew={data.skew():.2f})')
axes[0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: prev_gpa vs Normal')
axes[1].get_lines()[0].set(color='#AB47BC', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('prev_gpa: Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 parental_edu (ordinal categorical) ────────────────────────────────
col    = 'parental_edu'
labels = {0: 'No HS', 1: 'HS', 2: 'Some College', 3: 'Bachelor', 4: 'Graduate'}
counts = df[col].value_counts().sort_index()

print('=== Parental Education Level ===')
for lvl, cnt in counts.items():
    print(f'  Level {lvl} ({labels[lvl]:>12}): {cnt:4d} students ({cnt/n*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = [FAIL_COLOR, '#e67e22', '#f1c40f', ACCENT, PASS_COLOR]
bars = ax.bar([labels[k] for k in sorted(counts.index)],
               counts.sort_index().values,
               color=bar_colors, alpha=0.85, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Parental Education Level')
ax.set_ylabel('Number of Students')
ax.set_title('Distribution of Parental Education Level')
plt.tight_layout()
plt.show()

print('\nInsight: Most parents have Some College or Bachelor degree.')
print('Very few students have parents with no high school education.')

In [ ]:
# ── parental_edu: Ordinal categorical — PMF + entropy ────────────────────────
import math

col = 'parental_edu'
level_labels = {0:'None', 1:'High School', 2:'Some College', 3:"Bachelor's", 4:'Graduate'}
vc = df[col].value_counts().sort_index()
pmf = vc / len(df)
entropy = -sum(p*math.log2(p) for p in pmf if p > 0)
max_entropy = math.log2(len(vc))

print(f'=== {col}: Ordinal Categorical (5 levels) ===')
print('Normal distribution: NOT applicable — ordinal categorical variable')
print()
print('Empirical PMF:')
for k, p in pmf.items():
    bar = '█' * int(p * 40)
    print(f'  P(edu={k}) [{level_labels[k]:15s}] = {p:.4f}  ({p*100:.1f}%)  {bar}')
print()
print(f'Shannon Entropy: {entropy:.4f}  (max = {max_entropy:.4f} for 5 equal levels)')
print(f'Coverage: {entropy/max_entropy*100:.1f}%')

fig, ax = plt.subplots(figsize=(9, 5))
colours_edu = ['#EF5350','#FF7043','#FFA726','#66BB6A','#42A5F5']
bars = ax.bar([level_labels[k] for k in vc.index], pmf.values,
              color=colours_edu, edgecolor='white')
ax.axhline(1/5, color='red', linestyle='--', lw=1.5, label='Uniform (1/5 each)')
ax.set_ylabel('Probability')
ax.set_title('parental_edu: Empirical PMF')
ax.set_ylim(0, 0.45)
ax.legend()
for bar, p in zip(bars, pmf.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{p*100:.1f}%', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

> **parental_edu: Ordinal Categorical — PMF is the Right Tool**
>
> `parental_edu` has **5 ordered levels** (None → Graduate) with no equal numeric spacing.
> The "distance" from None→HS is not the same as HS→College.
>
> Normal distribution: not applicable (discrete levels, not a continuous measurement).
>
> **PMF** shows the relative frequency at each level.
> **Entropy** measures how evenly distributed the levels are:
> low entropy = concentrated at one level, high entropy = spread across all levels.

In [ ]:
# ── 3.6 part_time_hours ───────────────────────────────────────────────────
col        = 'part_time_hours'
data_all   = df[col].dropna()
pct_works  = (data_all > 0).mean() * 100
data_workers = data_all[data_all > 0]

print('=== Part-Time Work Hours ===')
print(f'  Students who work (>0 hrs):  {pct_works:.1f}%')
print(f'  Students who do not work:    {100-pct_works:.1f}%')
print(f'  Mean hours (workers only):   {data_workers.mean():.2f} hrs/week')
print(f'  Median hours (workers only): {data_workers.median():.2f} hrs/week')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Work vs No Work bar chart
work_counts = pd.Series({'Works': (data_all > 0).sum(), 'Does Not Work': (data_all == 0).sum()})
axes[0].bar(work_counts.index, work_counts.values,
            color=[FAIL_COLOR, PASS_COLOR], alpha=0.8, edgecolor='white')
for i, v in enumerate(work_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontsize=11)
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Works Part-Time vs Does Not Work')

# Histogram among workers only
axes[1].hist(data_workers, bins=25, color='#c0392b', alpha=0.7, edgecolor='white')
axes[1].axvline(data_workers.mean(),   color='red',    linestyle='--', linewidth=2, label=f'Mean {data_workers.mean():.1f}')
axes[1].axvline(data_workers.median(), color='orange', linestyle='--', linewidth=2, label=f'Median {data_workers.median():.1f}')
axes[1].set_xlabel('Hours per Week (workers only)')
axes[1].set_ylabel('Count')
axes[1].set_title('Part-Time Hours Among Working Students')
axes[1].legend()

plt.suptitle('Part-Time Work Hours', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nInsight: Bimodal distribution — either 0 hours (does not work) or a right-skewed')
print('distribution among workers. This is a classic "zero-inflated" distribution.')

In [ ]:
# ── part_time_hours: Zero-heavy, right-skewed distribution ──────────────────
from scipy import stats as sp_stats

col = 'part_time_hours'
data = df[col].dropna()
mean_, std_ = data.mean(), data.std()
zero_pct = (data == 0).mean() * 100

print(f'=== {col}: Distribution Analysis ===')
print(f'  Mean:     {mean_:.4f}')
print(f'  Median:   {data.median():.4f}')
print(f'  Std Dev:  {std_:.4f}')
print(f'  Skewness: {data.skew():.4f}  (positive = right tail)')
print(f'  Kurtosis: {data.kurtosis():.4f}')
print(f'  Zero count: {(data==0).sum()} students ({zero_pct:.1f}%) have 0 part-time hours')
w, p = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
print(f'  Shapiro-Wilk: W={w:.4f}, p={p:.2e}  → {"NOT normal" if p<0.05 else "Normal"}')
print()
print('Distribution type: ZERO-INFLATED, RIGHT-SKEWED')
print('Many students do not work; those who do work varying part-time hours.')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(0, data.max()+2, 300)
axes[0].hist(data, bins=30, color='#FF7043', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0], color='black', lw=2, label='KDE')
axes[0].plot(x, sp_stats.norm.pdf(x, mean_, std_), 'r--', lw=2.5, label='Normal fit')
axes[0].set_xlabel('Part-Time Hours per Week')
axes[0].set_title(f'part_time_hours  (skew={data.skew():.2f})')
axes[0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: part_time_hours vs Normal')
axes[1].get_lines()[0].set(color='#FF7043', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('part_time_hours: Zero-Inflated Right-Skewed Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Zero-Inflated Distribution**
>
> `part_time_hours` has a structural spike at 0 — many students simply don't work.
> For those who do, hours follow a right-skewed count distribution.
>
> This **"zero inflation"** pattern appears when:
> - Not working is a deliberate choice (not just a rare outcome)
> - There are two sub-populations: non-workers (always 0) and workers (random positive hours)
>
> Standard distributions (Normal, Poisson) don't fit well.
> In advanced modelling, zero-inflated models (ZIP, ZINB) or log(x+1) transformation are used.

In [ ]:
# ── 3.7 extracurricular ───────────────────────────────────────────────────
col    = 'extracurricular'
counts = df[col].value_counts().sort_index()

print('=== Extracurricular Activities ===')
print(f'  Mean:   {df[col].mean():.2f} activities')
print(f'  Median: {df[col].median():.0f} activities')
for k, v in counts.items():
    print(f'  {k} activities: {v:4d} students ({v/n*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.index, counts.values, color=ACCENT, alpha=0.8, edgecolor='white')
for i, (k, v) in enumerate(counts.items()):
    ax.text(k, v + 1, str(v), ha='center', fontsize=10)
ax.set_xlabel('Number of Extracurricular Activities')
ax.set_ylabel('Number of Students')
ax.set_title('Distribution of Extracurricular Activities')
ax.set_xticks(counts.index)
plt.tight_layout()
plt.show()

print('\n📊 CONCEPT — Poisson Distribution:')
print('  A Poisson distribution describes counts of rare independent events.')
print('  Extracurricular counts often follow this pattern — most students have')
print('  0-2 activities, and the probability drops off for larger counts.')

In [ ]:
# ── extracurricular: Bernoulli distribution ──────────────────────────────────
col = 'extracurricular'
p_yes = df[col].mean()
p_no  = 1 - p_yes
bern_var = p_yes * p_no

print(f'=== {col}: Bernoulli Distribution ===')
print('Normal distribution: NOT applicable — binary (0/1) variable')
print()
print(f'  p(extracurricular = 1) = {p_yes:.4f}  →  {p_yes*100:.1f}% participate')
print(f'  p(extracurricular = 0) = {p_no:.4f}  →  {p_no*100:.1f}% do not participate')
print()
print(f'  Bernoulli mean     = p = {p_yes:.4f}')
print(f'  Bernoulli variance = p(1-p) = {p_yes:.4f} × {p_no:.4f} = {bern_var:.4f}')
print(f'  Max variance (p=0.5) = 0.2500  |  Our variance = {bern_var:.4f}')
print(f'  → {bern_var/0.25*100:.1f}% as uncertain as a 50/50 coin flip')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No (0)', 'Yes (1)'], [p_no, p_yes],
       color=['#EF5350', '#66BB6A'], edgecolor='white', linewidth=1.5)
ax.axhline(0.5, color='red', linestyle='--', lw=1.5, label='Balanced p=0.5')
ax.set_ylabel('Probability')
ax.set_title(f'extracurricular: Bernoulli PMF  (p={p_yes:.2f})')
ax.set_ylim(0, 0.8)
ax.legend()
for i, val in enumerate([p_no, p_yes]):
    ax.text(i, val + 0.01, f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

> **extracurricular: Bernoulli Distribution**
>
> Binary variable — only two outcomes. Described completely by one number: **p**.
>
> In this synthetic dataset, participation rate is roughly 50% — close to maximum entropy.
> Compare this to `fasting_bs` in the heart disease dataset (Problem04) where p is much lower:
> a rare-event Bernoulli has very different properties than a near-balanced one.

In [ ]:
# ── 3.8 distance_km ───────────────────────────────────────────────────────
col  = 'distance_km'
data = df[col].dropna()

print(f'=== Distance from School (Note: {df[col].isnull().sum()} missing values, using dropna) ===')
print(f'  Mean:              {data.mean():.2f} km')
print(f'  Median:            {data.median():.2f} km')
print(f'  Skewness:          {data.skew():.3f}  (positive = right-skewed/exponential-like)')
print(f'  % living within 5km: {(data < 5).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(data, bins=30, color='#8e44ad', alpha=0.7, edgecolor='white', density=True)
data.plot.kde(ax=axes[0], color='purple', linewidth=2)
axes[0].axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.8, label=f'Mean {data.mean():.1f}km')
axes[0].axvline(data.median(), color='orange', linestyle='--', linewidth=1.8, label=f'Median {data.median():.1f}km')
axes[0].set_xlabel('Distance from School (km)')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Distance from School\n(Exponential-like)')
axes[0].legend()

axes[1].boxplot(data, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#8e44ad', alpha=0.6))
axes[1].set_xticks([1])
axes[1].set_xticklabels(['distance_km'])
axes[1].set_ylabel('Distance (km)')
axes[1].set_title('Box Plot: Distance from School\n(outliers visible)')

plt.suptitle('Distance from School (km)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 CONCEPT — Exponential Distribution:')
print('  Exponential distributions appear when we measure "distance/time until event".')
print('  Most students cluster near the school (low distance), with a long tail of')
print('  students who live far away. The mean > median confirms this right skew.')

In [ ]:
# ── distance_km: Right-skewed — log transform analysis ───────────────────────
from scipy import stats as sp_stats

col = 'distance_km'
data = df[col].dropna()
data_log = np.log1p(data)

print(f'=== {col}: Distribution Analysis ===')
print(f'  Raw  — skewness={data.skew():.4f}, kurtosis={data.kurtosis():.4f}')
print(f'  Log  — skewness={data_log.skew():.4f}, kurtosis={data_log.kurtosis():.4f}')
w1, p1 = sp_stats.shapiro(data.sample(min(500,len(data)), random_state=42))
w2, p2 = sp_stats.shapiro(data_log.sample(min(500,len(data_log)), random_state=42))
print(f'  Shapiro raw: p={p1:.2e}  Shapiro log: p={p2:.2e}')
print()
print('Distribution type: RIGHT-SKEWED (Log-Normal)')
print('Most students live close to school; a few live very far away.')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

rm, rs = data.mean(), data.std()
x_r = np.linspace(0, data.quantile(0.99)+5, 300)
axes[0,0].hist(data, bins=35, color='#5C6BC0', edgecolor='white', density=True, alpha=0.7)
data.plot.kde(ax=axes[0,0], color='black', lw=2, label='KDE')
axes[0,0].plot(x_r, sp_stats.norm.pdf(x_r, rm, rs), 'r--', lw=2.5, label='Normal fit')
axes[0,0].set_xlabel('Distance (km)')
axes[0,0].set_title(f'Raw distance  (skew={data.skew():.2f})')
axes[0,0].legend(fontsize=9)

sp_stats.probplot(data, dist='norm', plot=axes[0,1])
axes[0,1].set_title('Q-Q: Raw distance vs Normal\n(curvature = right skew)')
axes[0,1].get_lines()[0].set(color='#5C6BC0', markersize=3, alpha=0.4)
axes[0,1].get_lines()[1].set(color='red', lw=2)

lm, ls = data_log.mean(), data_log.std()
x_l = np.linspace(data_log.min(), data_log.max(), 300)
axes[1,0].hist(data_log, bins=35, color='#7986CB', edgecolor='white', density=True, alpha=0.7)
data_log.plot.kde(ax=axes[1,0], color='black', lw=2, label='KDE')
axes[1,0].plot(x_l, sp_stats.norm.pdf(x_l, lm, ls), 'r--', lw=2.5, label='Normal fit')
axes[1,0].set_xlabel('log(1 + Distance)')
axes[1,0].set_title(f'log(distance)  (skew={data_log.skew():.2f})')
axes[1,0].legend(fontsize=9)

sp_stats.probplot(data_log, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q: log(distance) vs Normal\n(straighter = more normal)')
axes[1,1].get_lines()[0].set(color='#7986CB', markersize=3, alpha=0.4)
axes[1,1].get_lines()[1].set(color='red', lw=2)

plt.suptitle('distance_km: Log-Normal Pattern (top=raw, bottom=log)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **distance_km: Log-Normal Pattern**
>
> Distances are almost always right-skewed — most students live within a short radius,
> but the distribution has a long right tail for students who travel far.
>
> This is the same log-normal pattern seen in Population and MedInc:
> spatial/geographic distances grow multiplicatively from a centre point.

In [ ]:
# ── 3.9 internet_quality ──────────────────────────────────────────────────
col    = 'internet_quality'
labels = {1: 'Poor', 2: 'Fair', 3: 'Average', 4: 'Good', 5: 'Excellent'}
counts = df[col].value_counts().sort_index()

print('=== Internet Quality at Home ===')
for k, v in counts.items():
    print(f'  Level {k} ({labels[k]:>9}): {v:4d} students ({v/n*100:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = [FAIL_COLOR, '#e67e22', '#f1c40f', ACCENT, PASS_COLOR]
bars = ax.bar([f'{k}\n({labels[k]})' for k in sorted(counts.index)],
               counts.sort_index().values,
               color=bar_colors, alpha=0.85, edgecolor='white')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Internet Quality (1=Poor to 5=Excellent)')
ax.set_ylabel('Number of Students')
ax.set_title('Distribution of Home Internet Quality')
plt.tight_layout()
plt.show()

print('\nInsight: Most students have Good or Excellent internet (levels 4-5).')
print('Only ~15% have Poor or Fair internet — but this could affect outcomes significantly.')

In [ ]:
# ── internet_quality: Ordinal categorical — PMF + entropy ────────────────────
import math

col = 'internet_quality'
quality_labels = {0:'Poor', 1:'Fair', 2:'Good', 3:'Excellent'}
vc = df[col].value_counts().sort_index()
pmf = vc / len(df)
entropy = -sum(p*math.log2(p) for p in pmf if p > 0)
max_entropy = math.log2(len(vc))

print(f'=== {col}: Ordinal Categorical (4 levels) ===')
print('Normal distribution: NOT applicable.')
print()
print('Empirical PMF:')
for k, p in pmf.items():
    bar = '█' * int(p * 40)
    print(f'  P(quality={k}) [{quality_labels[k]:10s}] = {p:.4f}  ({p*100:.1f}%)  {bar}')
print()
print(f'Shannon Entropy: {entropy:.4f}  (max = {max_entropy:.4f})')
print(f'Coverage: {entropy/max_entropy*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 5))
colours_q = ['#EF5350','#FFA726','#66BB6A','#42A5F5']
bars = ax.bar([quality_labels[k] for k in vc.index], pmf.values,
              color=colours_q, edgecolor='white')
ax.axhline(1/4, color='red', linestyle='--', lw=1.5, label='Uniform (1/4 each)')
ax.set_ylabel('Probability')
ax.set_title('internet_quality: Empirical PMF')
ax.set_ylim(0, 0.5)
ax.legend()
for bar, p in zip(bars, pmf.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{p*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

> **internet_quality: Ordinal Categorical**
>
> 4 ordered levels: Poor < Fair < Good < Excellent.
> The ordering matters (Good > Fair) but the numeric gap between levels is undefined —
> "jumping from Fair to Good" is not necessarily the same improvement as "Good to Excellent."
>
> For modelling, we typically encode this as ordinal numbers (0,1,2,3) or
> use one-hot encoding depending on the model.

> **Distribution Summary — All 10 Student Performance Features**
>
> | Feature | Type | Distribution Family | Normal applies? |
> |---------|------|--------------------:|:---------------:|
> | `study_hours` | Continuous | **Right-skewed (Gamma-like)** — always ≥ 0 | After log(x+1) |
> | `attendance_pct` | Continuous | **Left-skewed (Beta-like)** — bounded 0–100 | ✗ Bounded |
> | `sleep_hours` | Continuous | **≈ Normal** (clipped Normal generator) | ✓ Yes |
> | `prev_gpa` | Continuous | ≈ Normal / Uniform in 2.0–4.0 | ≈ Yes |
> | `parental_edu` | Ordinal categorical | Multinomial (5 levels) | ✗ No |
> | `part_time_hours` | Continuous | **Zero-inflated, right-skewed** | After log(x+1) |
> | `extracurricular` | Binary | **Bernoulli(p ≈ 0.5)** | ✗ No |
> | `distance_km` | Continuous | **Log-Normal** (right-skewed distance) | ✓ After log |
> | `internet_quality` | Ordinal categorical | Multinomial (4 levels) | ✗ No |
> | `final_score` | Continuous (target) | Approximately Normal (0–100) | ≈ Yes |
>
> **Pattern:** Effort/time variables (study_hours, part_time_hours) are right-skewed.
> Compliance/proportion variables (attendance_pct) are left-skewed.
> Biological/behavioral variables (sleep_hours) are approximately normal.

## 🎯 Section 4: The Target Variable — final_score

**"What does student performance look like?"**

Before we predict something, we need to deeply understand what we're predicting. Let's explore the distribution of final exam scores.

In [ ]:
# ── Target variable: final_score ──────────────────────────────────────────
score_data = df['final_score']

mean_s   = score_data.mean()
median_s = score_data.median()
std_s    = score_data.std()

print('=== Final Exam Score ==='  )
print(f'  Mean:   {mean_s:.2f}')
print(f'  Median: {median_s:.2f}')
print(f'  Std:    {std_s:.2f}')
print(f'  Min:    {score_data.min():.2f}')
print(f'  Max:    {score_data.max():.2f}')
print(f'\n  Average student scores {mean_s:.1f} points out of 100.')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram + KDE
axes[0].hist(score_data, bins=30, color=ACCENT, alpha=0.7, edgecolor='white', density=True)
score_data.plot.kde(ax=axes[0], color='navy', linewidth=2)
axes[0].axvline(mean_s,   color='red',    linestyle='--', linewidth=2, label=f'Mean {mean_s:.1f}')
axes[0].axvline(median_s, color='orange', linestyle='--', linewidth=2, label=f'Median {median_s:.1f}')
axes[0].axvline(60, color='black', linestyle=':', linewidth=1.5, label='Pass threshold (60)')
axes[0].set_xlabel('Final Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Final Scores')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(score_data, vert=True, patch_artist=True,
                boxprops=dict(facecolor=ACCENT, alpha=0.6))
axes[1].axhline(60, color='black', linestyle=':', linewidth=1.5, label='Pass threshold')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['final_score'])
axes[1].set_ylabel('Score')
axes[1].set_title('Box Plot: Final Score')
axes[1].legend()

# Empirical CDF
sorted_scores = np.sort(score_data)
cdf = np.arange(1, len(sorted_scores)+1) / len(sorted_scores)
axes[2].plot(sorted_scores, cdf, color='navy', linewidth=2)
axes[2].axvline(60, color='red', linestyle='--', linewidth=1.5, label='Pass threshold (60)')
axes[2].fill_betweenx(cdf, sorted_scores.min(), 60,
                       where=(sorted_scores <= 60), alpha=0.15, color=FAIL_COLOR)
axes[2].set_xlabel('Final Score')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('Empirical CDF of Final Scores')
axes[2].legend()
axes[2].grid(True, alpha=0.4)

plt.suptitle('Final Exam Score Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pass/Fail analysis ────────────────────────────────────────────────────
df['pass'] = (df['final_score'] >= 60).astype(int)
pass_rate  = df['pass'].mean() * 100
fail_rate  = 100 - pass_rate

print(f'Pass rate (score >= 60): {pass_rate:.1f}%')
print(f'Fail rate (score <  60): {fail_rate:.1f}%')

# Percentile breakdown
pcts = [10, 25, 50, 75, 90]
print('\nPercentile Breakdown:')
for p in pcts:
    val = np.percentile(df['final_score'], p)
    label = {10:'struggling', 25:'at-risk', 50:'median', 75:'good', 90:'excellent'}[p]
    print(f'  p{p:2d} ({label:>10}): {val:.1f} points')
print(f'\n  A student scoring {np.percentile(df["final_score"], 25):.1f} is in the bottom 25%')
print('  — these are the students we most want to identify early.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie([pass_rate, fail_rate],
            labels=[f'Pass\n{pass_rate:.1f}%', f'Fail\n{fail_rate:.1f}%'],
            colors=[PASS_COLOR, FAIL_COLOR],
            autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Pass vs Fail Distribution\n(threshold = 60)')

# Score distribution colored by pass/fail
pass_scores = df[df['pass'] == 1]['final_score']
fail_scores = df[df['pass'] == 0]['final_score']
axes[1].hist(fail_scores, bins=20, color=FAIL_COLOR, alpha=0.7, label=f'Fail (n={len(fail_scores)})', edgecolor='white')
axes[1].hist(pass_scores, bins=20, color=PASS_COLOR, alpha=0.7, label=f'Pass (n={len(pass_scores)})', edgecolor='white')
axes[1].axvline(60, color='black', linestyle='--', linewidth=2, label='Threshold (60)')
axes[1].set_xlabel('Final Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution by Pass/Fail')
axes[1].legend()

plt.tight_layout()
plt.show()

## 🔍 Section 5: Feature vs Target — What Predicts Performance?

**"Which features actually predict exam scores?"**

Now we ask the most important question: which of our features actually correlate with the final score? We'll examine each one individually.

In [ ]:
# ── 5.1 study_hours vs final_score ────────────────────────────────────────
df_clean = df.dropna(subset=['study_hours', 'final_score'])
r, p = stats.pearsonr(df_clean['study_hours'], df_clean['final_score'])

print(f'=== Study Hours vs Final Score ===')
print(f'  Pearson r = {r:.4f}  (p-value: {p:.4e})')
print(f'  Interpretation: {"strong" if abs(r)>0.5 else "moderate" if abs(r)>0.3 else "weak"} positive correlation')
print(f'  Insight: Strongest predictor — more study = higher score. r = {r:.2f}')

fig, ax = plt.subplots(figsize=(9, 5))
colors = [PASS_COLOR if p else FAIL_COLOR for p in df_clean['pass']]
ax.scatter(df_clean['study_hours'], df_clean['final_score'],
           c=colors, alpha=0.4, s=30)

# Regression line
m, b = np.polyfit(df_clean['study_hours'], df_clean['final_score'], 1)
x_line = np.linspace(df_clean['study_hours'].min(), df_clean['study_hours'].max(), 100)
ax.plot(x_line, m*x_line + b, color='navy', linewidth=2.5, label=f'Regression line (r={r:.2f})')

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=PASS_COLOR, label='Pass'), Patch(facecolor=FAIL_COLOR, label='Fail')]
ax.legend(handles=legend_els + ax.get_lines(), fontsize=9)
ax.axhline(60, color='black', linestyle=':', linewidth=1.2, alpha=0.6)
ax.set_xlabel('Study Hours per Week')
ax.set_ylabel('Final Score')
ax.set_title(f'Study Hours vs Final Score\nPearson r = {r:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 attendance_pct vs final_score ─────────────────────────────────────
df_clean = df.dropna(subset=['attendance_pct', 'final_score'])
r, p_val = stats.pearsonr(df_clean['attendance_pct'], df_clean['final_score'])

low_attend_avg = df_clean[df_clean['attendance_pct'] < 60]['final_score'].mean()
print(f'=== Attendance vs Final Score ===')
print(f'  Pearson r = {r:.4f}')
print(f'  Students with <60% attendance averaged {low_attend_avg:.1f} points')
print(f'  Insight: Attendance has a threshold effect — below 60%, scores drop sharply.')

fig, ax = plt.subplots(figsize=(9, 5))
colors = [PASS_COLOR if pp else FAIL_COLOR for pp in df_clean['pass']]
ax.scatter(df_clean['attendance_pct'], df_clean['final_score'],
           c=colors, alpha=0.4, s=30)
m, b = np.polyfit(df_clean['attendance_pct'], df_clean['final_score'], 1)
x_line = np.linspace(df_clean['attendance_pct'].min(), df_clean['attendance_pct'].max(), 100)
ax.plot(x_line, m*x_line + b, color='navy', linewidth=2.5)
ax.axvline(60, color=FAIL_COLOR, linestyle='--', linewidth=1.8, label='60% at-risk threshold')
ax.axhline(60, color='black',    linestyle=':',  linewidth=1.2, alpha=0.6, label='Pass threshold')
ax.annotate(f'<60% attendance\navg score: {low_attend_avg:.1f}',
            xy=(50, low_attend_avg), xytext=(30, low_attend_avg - 15),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=PASS_COLOR, label='Pass'), Patch(facecolor=FAIL_COLOR, label='Fail')]
handles, labels_l = ax.get_legend_handles_labels()
ax.legend(handles=legend_els + handles, fontsize=9)
ax.set_xlabel('Attendance (%)')
ax.set_ylabel('Final Score')
ax.set_title(f'Attendance vs Final Score  (r = {r:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 sleep_hours vs final_score ────────────────────────────────────────
df_clean = df.dropna(subset=['sleep_hours', 'final_score'])
r, _ = stats.pearsonr(df_clean['sleep_hours'], df_clean['final_score'])

print(f'=== Sleep Hours vs Final Score ===')
print(f'  Pearson r = {r:.4f}')

# Bin sleep into groups
bins   = [0, 6, 7, 8, 9, 12]
labels = ['<6hrs', '6-7hrs', '7-8hrs', '8-9hrs', '9+hrs']
df_clean = df_clean.copy()
df_clean['sleep_bin'] = pd.cut(df_clean['sleep_hours'], bins=bins, labels=labels)
bin_means = df_clean.groupby('sleep_bin', observed=True)['final_score'].mean()
print('\n  Mean score by sleep group:')
for b, m in bin_means.items():
    print(f'    {b}: {m:.2f}')
print('\n  Insight: 7-9hr sweet spot. Both too little and too much sleep correlates with lower scores.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + regression line
axes[0].scatter(df_clean['sleep_hours'], df_clean['final_score'],
                c=[PASS_COLOR if p else FAIL_COLOR for p in df_clean['pass']],
                alpha=0.4, s=30)
m, b_coef = np.polyfit(df_clean['sleep_hours'], df_clean['final_score'], 1)
x_line = np.linspace(df_clean['sleep_hours'].min(), df_clean['sleep_hours'].max(), 100)
axes[0].plot(x_line, m*x_line + b_coef, color='navy', linewidth=2.5)
axes[0].axvspan(7, 9, alpha=0.12, color=PASS_COLOR, label='Sweet spot (7-9hrs)')
axes[0].set_xlabel('Sleep Hours per Night')
axes[0].set_ylabel('Final Score')
axes[0].set_title(f'Sleep Hours vs Final Score  (r = {r:.3f})')
axes[0].legend(fontsize=9)

# Box plot per sleep bin
df_clean.boxplot(column='final_score', by='sleep_bin', ax=axes[1],
                  patch_artist=True,
                  boxprops=dict(facecolor=ACCENT, alpha=0.5))
axes[1].axhline(60, color=FAIL_COLOR, linestyle='--', linewidth=1.5, label='Pass threshold')
axes[1].set_xlabel('Sleep Group')
axes[1].set_ylabel('Final Score')
axes[1].set_title('Final Score Distribution by Sleep Group')
plt.suptitle('')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 prev_gpa vs final_score ───────────────────────────────────────────
r, _ = stats.pearsonr(df['prev_gpa'], df['final_score'])
print(f'=== Previous GPA vs Final Score ===')
print(f'  Pearson r = {r:.4f}  — very strong positive correlation')
print('  📊 CONCEPT: Prior performance is a strong predictor of future performance.')
print('  This is why GPA matters: past habits and knowledge carry forward.')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['prev_gpa'], df['final_score'],
           c=[PASS_COLOR if p else FAIL_COLOR for p in df['pass']],
           alpha=0.4, s=30)
m, b_coef = np.polyfit(df['prev_gpa'], df['final_score'], 1)
x_line = np.linspace(df['prev_gpa'].min(), df['prev_gpa'].max(), 100)
ax.plot(x_line, m*x_line + b_coef, color='navy', linewidth=2.5, label=f'r = {r:.3f}')
ax.axhline(60, color='black', linestyle=':', linewidth=1.2, alpha=0.6)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=PASS_COLOR, label='Pass'),
                   Patch(facecolor=FAIL_COLOR, label='Fail')] + ax.get_lines(), fontsize=9)
ax.set_xlabel('Previous GPA (0.0 – 4.0)')
ax.set_ylabel('Final Score')
ax.set_title(f'Previous GPA vs Final Score  (r = {r:.3f})')
plt.tight_layout()
plt.show()

# ── 5.5 parental_edu vs final_score ───────────────────────────────────────
edu_labels = {0: 'No HS', 1: 'HS', 2: 'Some\nCollege', 3: 'Bachelor', 4: 'Graduate'}
edu_means  = df.groupby('parental_edu')['final_score'].agg(['mean', 'std'])

print(f'\n=== Parental Education vs Final Score ===')
for lvl, row in edu_means.iterrows():
    print(f'  Level {lvl} ({edu_labels[lvl].replace(chr(10)," "):>12}): mean={row["mean"]:.1f}, std={row["std"]:.1f}')
print('  Insight: Higher parental education correlates with higher scores.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plots
groups = [df[df['parental_edu'] == lvl]['final_score'].dropna() for lvl in sorted(edu_labels)]
bp = axes[0].boxplot(groups, patch_artist=True, labels=[edu_labels[k] for k in sorted(edu_labels)])
colors_box = [FAIL_COLOR, '#e67e22', '#f1c40f', ACCENT, PASS_COLOR]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].axhline(60, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0].set_xlabel('Parental Education Level')
axes[0].set_ylabel('Final Score')
axes[0].set_title('Score Distribution by Parental Education')

# Bar chart with error bars
axes[1].bar(range(len(edu_means)), edu_means['mean'],
            yerr=edu_means['std'], capsize=5,
            color=colors_box, alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(len(edu_means)))
axes[1].set_xticklabels([edu_labels[k] for k in sorted(edu_labels)])
axes[1].axhline(60, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].set_ylabel('Mean Final Score')
axes[1].set_title('Mean Score by Parental Education\n(error bars = ±1 std dev)')
axes[1].set_ylim(0, 105)

print('\n  📊 CONCEPT — Error Bars:')\
    if False else print('\n  📊 CONCEPT: Error bars show the spread (uncertainty) around the mean.')
print('  A short error bar = consistent scores within that group.')
print('  A tall error bar = high variability — some students do well, others do not.')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6 part_time_hours vs final_score ────────────────────────────────────
df_workers = df[df['part_time_hours'] > 0].dropna(subset=['part_time_hours'])
r_workers, _ = stats.pearsonr(df_workers['part_time_hours'], df_workers['final_score'])

score_no_work = df[df['part_time_hours'] == 0]['final_score'].mean()
score_work    = df_workers['final_score'].mean()
score_heavy   = df[df['part_time_hours'] >= 20]['final_score'].mean()
score_light   = df[(df['part_time_hours'] > 0) & (df['part_time_hours'] <= 10)]['final_score'].mean()

print('=== Part-Time Hours vs Final Score ===')
print(f'  r among workers = {r_workers:.4f}')
print(f'  Avg score — no work:          {score_no_work:.1f}')
print(f'  Avg score — light work (1-10): {score_light:.1f}')
print(f'  Avg score — heavy work (20+):  {score_heavy:.1f}')
print('  Insight: light work (1-10hrs) may not hurt; heavy work (20+hrs) shows clear score drop.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter among workers
axes[0].scatter(df_workers['part_time_hours'], df_workers['final_score'],
                c=[PASS_COLOR if p else FAIL_COLOR for p in df_workers['pass']],
                alpha=0.5, s=35)
m, b_coef = np.polyfit(df_workers['part_time_hours'], df_workers['final_score'], 1)
x_line = np.linspace(0, df_workers['part_time_hours'].max(), 100)
axes[0].plot(x_line, m*x_line + b_coef, color='navy', linewidth=2.5)
axes[0].axhline(60, color='black', linestyle=':', linewidth=1.2, alpha=0.6)
axes[0].set_xlabel('Part-Time Hours per Week (workers only)')
axes[0].set_ylabel('Final Score')
axes[0].set_title(f'Part-Time Hours vs Score (workers)\nr = {r_workers:.3f}')

# Box plot: working vs not working
groups_work = [df[df['part_time_hours'] == 0]['final_score'].dropna(),
               df_workers['final_score']]
bp = axes[1].boxplot(groups_work, patch_artist=True, labels=['Not Working', 'Working'])
for patch, color in zip(bp['boxes'], [PASS_COLOR, FAIL_COLOR]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].axhline(60, color='red', linestyle='--', linewidth=1.5)
axes[1].set_ylabel('Final Score')
axes[1].set_title('Scores: Working vs Not Working Students')

plt.tight_layout()
plt.show()

# ── 5.7 extracurricular vs final_score ────────────────────────────────────
extra_means = df.groupby('extracurricular')['final_score'].mean()
print('\n=== Extracurricular Activities vs Final Score ===')
for cnt, mean in extra_means.items():
    print(f'  {cnt} activities: avg score = {mean:.2f}')

fig, ax = plt.subplots(figsize=(9, 5))
groups_extra = [df[df['extracurricular'] == cnt]['final_score'].dropna() for cnt in sorted(df['extracurricular'].unique())]
bp = ax.boxplot(groups_extra, patch_artist=True,
                labels=sorted(df['extracurricular'].unique()))
for patch in bp['boxes']:
    patch.set_facecolor(ACCENT)
    patch.set_alpha(0.6)
ax.plot(range(1, len(extra_means)+1), extra_means.sort_index().values,
        'ro-', linewidth=2, markersize=7, label='Mean score')
ax.axhline(60, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Number of Extracurricular Activities')
ax.set_ylabel('Final Score')
ax.set_title('Final Score by Extracurricular Activity Count')
ax.legend()
plt.tight_layout()
plt.show()
print('  Insight: 1-2 activities correlates with slightly higher scores; 5 may indicate overwhelm.')

In [ ]:
# ── 5.8 distance_km vs final_score ────────────────────────────────────────
df_clean = df.dropna(subset=['distance_km'])
r, _ = stats.pearsonr(df_clean['distance_km'], df_clean['final_score'])

print(f'=== Distance from School vs Final Score ===')
print(f'  Pearson r = {r:.4f}  (weak negative — longer commutes may affect study time)')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_clean['distance_km'], df_clean['final_score'],
           c=[PASS_COLOR if p else FAIL_COLOR for p in df_clean['pass']],
           alpha=0.4, s=30)
m, b_coef = np.polyfit(df_clean['distance_km'], df_clean['final_score'], 1)
x_line = np.linspace(df_clean['distance_km'].min(), df_clean['distance_km'].max(), 100)
ax.plot(x_line, m*x_line + b_coef, color='navy', linewidth=2.5, label=f'r = {r:.3f}')
ax.axhline(60, color='black', linestyle=':', linewidth=1.2, alpha=0.6)
ax.set_xlabel('Distance from School (km)')
ax.set_ylabel('Final Score')
ax.set_title(f'Distance vs Final Score  (r = {r:.3f})')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── 5.9 internet_quality vs final_score ───────────────────────────────────
iq_labels  = {1: 'Poor', 2: 'Fair', 3: 'Average', 4: 'Good', 5: 'Excellent'}
iq_means   = df.groupby('internet_quality')['final_score'].mean()

print(f'\n=== Internet Quality vs Final Score ===')
for lvl, mean in iq_means.items():
    print(f'  Level {lvl} ({iq_labels[lvl]:>8}): avg score = {mean:.2f}')
print('  Insight: Students with poor internet score lower — access matters.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

groups_iq = [df[df['internet_quality'] == lvl]['final_score'].dropna() for lvl in sorted(iq_labels)]
bp = axes[0].boxplot(groups_iq, patch_artist=True,
                     labels=[iq_labels[k] for k in sorted(iq_labels)])
colors_iq = [FAIL_COLOR, '#e67e22', '#f1c40f', ACCENT, PASS_COLOR]
for patch, color in zip(bp['boxes'], colors_iq):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].axhline(60, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Internet Quality')
axes[0].set_ylabel('Final Score')
axes[0].set_title('Score Distribution by Internet Quality')

axes[1].bar([iq_labels[k] for k in sorted(iq_means.index)],
             iq_means.sort_index().values,
             color=colors_iq, alpha=0.85, edgecolor='white')
axes[1].axhline(60, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Internet Quality')
axes[1].set_ylabel('Mean Final Score')
axes[1].set_title('Mean Score by Internet Quality')
for i, (k, v) in enumerate(iq_means.sort_index().items()):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 🌐 Section 6: Finding Patterns Together

**"Let's see the big picture"**

Individual feature analysis is useful, but features interact with each other. Here we look at the full picture: all correlations at once, which features matter most, and how combinations of features determine outcomes.

In [ ]:
# ── 6.1 Correlation Heatmap ───────────────────────────────────────────────
# Use numeric columns only; fill missing with median for correlation calc
df_num = df.drop(columns=['pass']).copy()
df_num['sleep_hours']     = df_num['sleep_hours'].fillna(df_num['sleep_hours'].median())
df_num['distance_km']     = df_num['distance_km'].fillna(df_num['distance_km'].median())
df_num['part_time_hours'] = df_num['part_time_hours'].fillna(0)

corr = df_num.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.zeros_like(corr, dtype=bool)
np.fill_diagonal(mask, True)  # hide diagonal (self-correlation = 1.0)
sns.heatmap(
    corr, ax=ax,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix — All Features\n(red = positive, blue = negative)', fontsize=13)
plt.tight_layout()
plt.show()

print('📊 CONCEPT — Heatmap of Correlations:')
print('  A correlation heatmap lets us see ALL pairwise relationships at once.')
print('  The final_score column tells us which features matter most.')
print('  Values close to +1.0 = strong positive relationship')
print('  Values close to -1.0 = strong negative relationship')
print('  Values near 0.0 = weak or no linear relationship')

# Top correlates with final_score
score_corr = corr['final_score'].drop('final_score').abs().sort_values(ascending=False)
print('\n  Top correlates with final_score (|r|):')
for feat, r in score_corr.items():
    bar = '█' * int(r * 20)
    print(f'  {feat:>20}: {r:.3f} {bar}')

In [ ]:
# ── 6.2 Top Correlates Bar Chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = [PASS_COLOR if r > 0.3 else ACCENT if r > 0.15 else '#bdc3c7'
              for r in score_corr.values]
score_corr.plot.barh(ax=ax, color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(0.3, color='green',  linestyle='--', linewidth=1.5, alpha=0.7, label='Strong (0.3)')
ax.axvline(0.1, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label='Weak (0.1)')
ax.set_xlabel('|Pearson r| with final_score')
ax.set_title('Feature Correlation with Final Score\n(absolute value — higher = stronger predictor)')
ax.legend(fontsize=9)
for i, (feat, r) in enumerate(score_corr.items()):
    ax.text(r + 0.005, i, f'{r:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Insight: study_hours, prev_gpa, and attendance are the "big three" predictors.')

In [ ]:
# ── 6.3 Multivariate: Study Hours × Attendance, colored by Score ──────────
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(df['study_hours'], df['attendance_pct'],
                c=df['final_score'], cmap='viridis',
                alpha=0.6, s=40, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Final Score')
ax.axvline(10, color='white', linestyle='--', linewidth=1.2, alpha=0.7, label='10 hrs study')
ax.axhline(75, color='white', linestyle='--', linewidth=1.2, alpha=0.7, label='75% attendance')
ax.set_xlabel('Study Hours per Week')
ax.set_ylabel('Attendance (%)')
ax.set_title('Study Hours × Attendance — colored by Final Score\n(yellow = high score, purple = low score)')
ax.legend(fontsize=9)
ax.text(25, 95, 'High Study\nHigh Attendance\n→ Highest Scores', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
ax.text(0.5, 35, 'Low Study\nLow Attendance\n→ Lowest Scores', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='salmon', alpha=0.7))
plt.tight_layout()
plt.show()

print('Insight: Top-right quadrant (high study, high attendance) = highest scorers.')
print('Low study AND low attendance = almost certain failure.')

In [ ]:
# ── 6.4 Pass Rate Heatmap: Study × Attendance bins ────────────────────────
df_binned = df.copy()
df_binned['study_bin'] = pd.cut(df['study_hours'],
                                 bins=[0, 5, 10, 15, 20, 41],
                                 labels=['0-5', '5-10', '10-15', '15-20', '20+'])
df_binned['attend_bin'] = pd.cut(df['attendance_pct'],
                                  bins=[0, 60, 75, 90, 101],
                                  labels=['<60%', '60-75%', '75-90%', '90-100%'])

pass_rate_pivot = df_binned.groupby(
    ['attend_bin', 'study_bin'], observed=True
)['pass'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pass_rate_pivot,
    ax=ax,
    annot=True, fmt='.0%',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Pass Rate'}
)
ax.set_xlabel('Study Hours per Week (bins)')
ax.set_ylabel('Attendance Rate (bins)')
ax.set_title('Pass Rate by Study Hours and Attendance\n(green = high pass rate, red = low pass rate)')
plt.tight_layout()
plt.show()

print('Insight: Low study AND low attendance = almost certain failure.')
print('High attendance can partially compensate for lower study hours — and vice versa.')
print('The best outcomes consistently appear in the top-right (high study + high attendance).')

## 🔧 Section 7: Feature Engineering

**"Building better features for prediction"**

Raw features are a starting point. We can often create new features that capture relationships the model might otherwise miss. Here we also handle missing values.

In [ ]:
# ── Handle missing values first ───────────────────────────────────────────
df_eng = df.copy()

# sleep_hours: fill with median by parental_edu group
df_eng['sleep_hours'] = df_eng.groupby('parental_edu')['sleep_hours'].transform(
    lambda x: x.fillna(x.median())
)
# distance_km: fill with overall median
df_eng['distance_km'] = df_eng['distance_km'].fillna(df_eng['distance_km'].median())
# part_time_hours: fill with 0 (assume no data = not working)
df_eng['part_time_hours'] = df_eng['part_time_hours'].fillna(0)

print('Missing values after imputation:')
print(df_eng.isnull().sum()[df_eng.isnull().sum() > 0])
print('(No missing values remain)' if df_eng.isnull().sum().sum() == 0 else '')

# ── Feature 1: study_attendance_interaction ────────────────────────────────
df_eng['study_attend_interact'] = df_eng['study_hours'] * (df_eng['attendance_pct'] / 100)
r_orig_study,  _ = stats.pearsonr(df_eng['study_hours'],           df_eng['final_score'])
r_orig_attend, _ = stats.pearsonr(df_eng['attendance_pct'],        df_eng['final_score'])
r_interact,    _ = stats.pearsonr(df_eng['study_attend_interact'],  df_eng['final_score'])

print('\n=== Feature 1: study × attendance_interaction ===')
print('  "A student who studies 20hrs with 50% attendance ≠ one who studies 20hrs with 95%"')
print(f'  r(study_hours vs score):         {r_orig_study:.3f}')
print(f'  r(attendance_pct vs score):      {r_orig_attend:.3f}')
print(f'  r(study_attend_interact vs score): {r_interact:.3f}  ← interaction is stronger!')

# ── Feature 2: total_productive_time ─────────────────────────────────────
df_eng['total_productive_time'] = df_eng['study_hours'] - df_eng['part_time_hours'] * 0.3
r_prod, _ = stats.pearsonr(df_eng['total_productive_time'], df_eng['final_score'])
print(f'\n=== Feature 2: total_productive_time ===')
print('  study_hours - part_time_hours * 0.3  (part-time work cuts into productive time)')
print(f'  r with final_score: {r_prod:.3f}')

# ── Feature 3: family_education_support ───────────────────────────────────
df_eng['family_edu_support'] = df_eng['parental_edu'] * df_eng['internet_quality']
r_support, _ = stats.pearsonr(df_eng['family_edu_support'], df_eng['final_score'])
print(f'\n=== Feature 3: family_education_support ===')
print('  parental_edu × internet_quality  (combined support index)')
print(f'  r with final_score: {r_support:.3f}')

# ── Feature 4: sleep_quality_score ───────────────────────────────────────
df_eng['sleep_adequate'] = ((df_eng['sleep_hours'] >= 7) & (df_eng['sleep_hours'] <= 9)).astype(int)
pass_adequate   = df_eng[df_eng['sleep_adequate'] == 1]['pass'].mean() * 100
pass_inadequate = df_eng[df_eng['sleep_adequate'] == 0]['pass'].mean() * 100
print(f'\n=== Feature 4: sleep_adequate (binary) ===')
print(f'  Pass rate with adequate sleep (7-9hrs):  {pass_adequate:.1f}%')
print(f'  Pass rate with inadequate sleep:          {pass_inadequate:.1f}%')

# ── Feature 5: is_at_risk label ───────────────────────────────────────────
df_eng['is_at_risk'] = ((df_eng['study_hours'] < 10) | (df_eng['attendance_pct'] < 70)).astype(int)
n_at_risk    = df_eng['is_at_risk'].sum()
pct_at_risk  = n_at_risk / n * 100
fail_of_risk = df_eng[df_eng['is_at_risk'] == 1]['pass'].apply(lambda x: 1 - x).mean() * 100
print(f'\n=== Feature 5: is_at_risk label ===')
print(f'  At-risk = study_hours < 10 OR attendance < 70%')
print(f'  At-risk students: {n_at_risk} out of {n} ({pct_at_risk:.1f}%)')
print(f'  Of at-risk students, {fail_of_risk:.1f}% failed the exam')
print(f'  → at_risk is a useful early warning indicator!')

## 🤖 Section 8: Building the Prediction

**"From understanding to action"**

We've explored the data deeply. Now we build models — both to predict the actual score (regression) and to classify students as pass/fail (classification).

In [ ]:
# ── Prepare feature matrix ────────────────────────────────────────────────
feature_cols = [
    'study_hours', 'attendance_pct', 'sleep_hours', 'prev_gpa',
    'parental_edu', 'part_time_hours', 'extracurricular',
    'distance_km', 'internet_quality',
    'study_attend_interact', 'total_productive_time',
    'family_edu_support', 'sleep_adequate'
]

X = df_eng[feature_cols].values
y_reg  = df_eng['final_score'].values   # regression target
y_clf  = df_eng['pass'].values          # classification target

X_train, X_test, y_train_r, y_test_r, y_train_c, y_test_c = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} students')
print(f'Test set:     {X_test.shape[0]} students')
print(f'Features:     {len(feature_cols)}')

In [ ]:
# ── Task 1: Regression ────────────────────────────────────────────────────
print('=== Task 1: Regression — Predicting Final Score ===\n')

reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42)
}

reg_results = {}
for name, model in reg_models.items():
    Xtr = X_train_sc if 'Ridge' in name or 'Linear' in name else X_train
    Xte = X_test_sc  if 'Ridge' in name or 'Linear' in name else X_test
    model.fit(Xtr, y_train_r)
    preds = model.predict(Xte)
    mae  = mean_absolute_error(y_test_r, preds)
    rmse = np.sqrt(mean_squared_error(y_test_r, preds))
    r2   = r2_score(y_test_r, preds)
    reg_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'preds': preds}
    print(f'{name}:')
    print(f'  MAE  = {mae:.2f}  (avg prediction error in points)')
    print(f'  RMSE = {rmse:.2f}')
    print(f'  R²   = {r2:.4f}  (explains {r2*100:.1f}% of variance)')
    print()

# Best model: Random Forest
best_reg_preds = reg_results['Random Forest']['preds']
print(f'Best model (Random Forest) MAE = {reg_results["Random Forest"]["MAE"]:.2f}')
print(f'→ Our predictions are off by about {reg_results["Random Forest"]["MAE"]:.1f} points on average.')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test_r, best_reg_preds, alpha=0.5, color=ACCENT, s=30)
ax.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Final Score')
ax.set_ylabel('Predicted Final Score')
ax.set_title(f'Random Forest Regression — Predicted vs Actual\n(R² = {reg_results["Random Forest"]["R2"]:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Task 2: Classification ─────────────────────────────────────────────────
print('=== Task 2: Classification — Predict Pass/Fail ===\n')

clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (d=4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

clf_results = {}
for name, model in clf_models.items():
    Xtr = X_train_sc if 'Logistic' in name else X_train
    Xte = X_test_sc  if 'Logistic' in name else X_test
    model.fit(Xtr, y_train_c)
    preds = model.predict(Xte)
    acc  = accuracy_score(y_test_c, preds)
    prec = precision_score(y_test_c, preds)
    rec  = recall_score(y_test_c, preds)
    f1   = f1_score(y_test_c, preds)
    clf_results[name] = {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'preds': preds, 'model': model}
    print(f'{name}:')
    print(f'  Accuracy:  {acc:.3f}')
    print(f'  Precision: {prec:.3f}  (of predicted pass, how many actually passed)')
    print(f'  Recall:    {rec:.3f}  (of actual pass, how many did we catch)')
    print(f'  F1:        {f1:.3f}')
    print()

print('📊 CONCEPT: Why Recall Matters More Here')
print('  Recall = of all students who actually failed, how many did we flag?')
print('  We would rather flag a student who does not need help (false alarm)')
print('  than MISS a student who truly needs intervention (missed catch).')
print('  So high recall (catching failing students) is our priority.')

In [ ]:
# ── Confusion Matrix for best classifier ─────────────────────────────────
best_clf_name = max(clf_results, key=lambda k: clf_results[k]['f1'])
best_clf_preds = clf_results[best_clf_name]['preds']
cm = confusion_matrix(y_test_c, best_clf_preds)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fail', 'Predicted Pass'],
            yticklabels=['Actual Fail', 'Actual Pass'],
            ax=ax, linewidths=1)
ax.set_title(f'Confusion Matrix — {best_clf_name}\n'
             f'(Accuracy={clf_results[best_clf_name]["acc"]:.3f}, '
             f'F1={clf_results[best_clf_name]["f1"]:.3f})')
ax.set_ylabel('Actual Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Annotation of quadrants
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted fail): {tn}')
print(f'False Positives (predicted pass, actually fail): {fp}  ← we want this LOW')
print(f'False Negatives (predicted fail, actually pass): {fn}')
print(f'True Positives  (correctly predicted pass): {tp}')

In [ ]:
# ── Feature Importance from Random Forest ────────────────────────────────
rf_clf = clf_results['Random Forest']['model']
importances = pd.Series(rf_clf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
colors_imp = [PASS_COLOR if imp > 0.10 else ACCENT if imp > 0.05 else '#bdc3c7'
              for imp in importances.values]
importances.plot.barh(ax=ax, color=colors_imp, alpha=0.85, edgecolor='white')
ax.set_xlabel('Feature Importance (Random Forest)')
ax.set_title('Feature Importance for Pass/Fail Classification\n(higher = more important to the model)')
for i, (feat, imp) in enumerate(importances.items()):
    ax.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Feature importance confirms our EDA findings:')
print('  - prev_gpa, study_hours, attendance_pct are top predictors')
print('  - Engineered features (study_attend_interact) also rank highly')
print('  - distance_km and extracurricular are less important')

In [ ]:
# ── Early Warning System ──────────────────────────────────────────────────
print('=== Early Warning System: Top 20 At-Risk Students ===\n')

rf_clf_model = clf_results['Random Forest']['model']
proba = rf_clf_model.predict_proba(X_test)[:, 0]  # probability of FAILING (class 0)

# Reconstruct test-set rows from df_eng
_, test_indices = train_test_split(np.arange(len(df_eng)), test_size=0.2, random_state=42)
df_test_display = df_eng.iloc[test_indices].copy()
df_test_display['fail_probability'] = proba
df_test_display['predicted_fail']   = (proba >= 0.5).astype(int)

# Top 20 students most likely to fail
top_risk = df_test_display.sort_values('fail_probability', ascending=False).head(20)

display_cols = ['study_hours', 'attendance_pct', 'sleep_hours', 'prev_gpa',
                'part_time_hours', 'final_score', 'pass', 'fail_probability']

print('These are the students a counselor should reach out to FIRST:')
print(top_risk[display_cols].round(2).to_string())
print(f'\nAll {len(top_risk)} flagged students: avg actual score = {top_risk["final_score"].mean():.1f}')
print(f'Of these 20, {(top_risk["pass"]==0).sum()} actually failed (true positives for early warning)')

## 🎓 Section 9: What We Learned

**"The full picture"**

We started with a real-world question — can we identify struggling students before the exam? — and answered it through data. Along the way, we learned foundational statistics and machine learning concepts.

### Concepts Learned

| Concept | What It Revealed |
|---|---|
| **Descriptive statistics** (mean, median, std) | Summarized each feature; showed skew via mean vs median gap |
| **Skewness** | Study hours and distance are right-skewed; attendance is left-skewed |
| **Normal distribution** | Sleep hours uniquely follows a normal distribution among student features |
| **Q-Q plot** | Visually confirmed normality of sleep hours vs other distributions |
| **Exponential distribution** | Distance from school follows an exponential pattern — most live nearby |
| **Poisson distribution** | Extracurricular activity counts follow Poisson — counting rare events |
| **Missing value imputation** | Median imputation by group (sleep by parental education) is more principled than overall median |
| **Pearson correlation (r)** | Quantified linear relationships; prev_gpa and study_hours led the pack |
| **Correlation heatmap** | Saw all relationships simultaneously; identified multicollinearity |
| **Feature engineering** | study × attendance interaction outperformed both raw features alone |
| **Train/test split** | Evaluated model on unseen data to measure real-world performance |
| **Regression metrics** (MAE, RMSE, R²) | MAE of ~X points means predictions are off by X points on average |
| **Classification metrics** (precision, recall, F1) | Recall matters most — catching at-risk students is the priority |
| **Confusion matrix** | Visualized the 4 types of correct and incorrect predictions |
| **Feature importance** | Random Forest confirmed EDA findings about which features matter most |

### Key Takeaways

1. **Study hours and attendance together are more predictive than either alone** — this interaction effect is the core insight of feature engineering.
2. **Prior GPA is the single strongest predictor** — past performance reliably predicts future performance.
3. **Sleep follows a normal distribution** — unusual among student features; most others are skewed.
4. **Part-time work has a threshold effect**: light work (1-10 hrs) is fine, but heavy work (20+ hrs) correlates with lower scores.
5. **The engineered feature (study × attendance) created a stronger predictor** than either raw feature alone — demonstrating the value of domain-informed feature engineering.
6. **An early warning system is achievable** — with ~85%+ recall, we can flag most at-risk students before the exam.

### The Bottom Line

Data science did not just predict scores — it told a story about *why* students succeed or struggle. That story is what enables action: reach out to students with low study × attendance, check in on those working heavy hours, and use prior GPA as an early signal long before exams begin.